In [ ]:
import pickle
import random as rd
from tqdm import tqdm
from typing import Tuple, Optional
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

from sklearn.metrics import f1_score
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer

import numpy as np
import pandas as pd

from datasets import load_dataset

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.normalizers import Lowercase
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.decoders import BPEDecoder, Replace

from transformers import PreTrainedTokenizerFast

In [16]:
dataset = load_dataset("mteb/emotion")
print(f"Dataset type: {type(dataset)}")
dataset

Dataset type: <class 'datasets.dataset_dict.DatasetDict'>


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 15956
    })
    validation: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 1988
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 1986
    })
})

In [17]:
train_dataset = dataset["train"].to_pandas()
val_dataset = dataset["validation"].to_pandas()
test_dataset = dataset['test'].to_pandas()
print(f"Splits dtype: {type(train_dataset), type(val_dataset), type(train_dataset)}")
print(len(train_dataset), len(val_dataset), len(test_dataset))

Splits dtype: (<class 'pandas.DataFrame'>, <class 'pandas.DataFrame'>, <class 'pandas.DataFrame'>)
15956 1988 1986


In [18]:
print("Random train row:")
print(train_dataset.sample(), end = '\n\n')
print("Random val row:")
print(val_dataset.sample(), end = '\n\n')
print("Random test row:")
print(test_dataset.sample())

Random train row:
                                                   text  label label_text
3378  i hear the word and i feel stronger and re ass...      1        joy

Random val row:
                                                   text  label label_text
1681  i was feeling extremely whiney and lonely and sad      0    sadness

Random test row:
                                                  text  label label_text
172  i have to admit i feel amused when i see the p...      1        joy


In [19]:
# Unique labels
train_dataset[["label", "label_text"]].drop_duplicates()

,label,label_text
0,0,sadness
2,3,anger
3,2,love
6,5,surprise
7,4,fear
8,1,joy


In [20]:
def get_label_text(label: int) -> str:
    label_data = train_dataset[["label", "label_text"]].drop_duplicates()
    for row in label_data.itertuples():
        if row.label == label:
            return row.label_text
    raise ValueError("There is no such label")

### Dummy model

In [67]:
dummy_clsf = DummyClassifier(strategy = 'most_frequent')
dummy_clsf

,"strategy strategy: {""most_frequent"", ""prior"", ""stratified"", ""uniform"", ""constant""}, default=""prior""Strategy to use to generate predictions.* ""most_frequent"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit`. The `predict_proba` method returns the matching one-hot encoded vector.* ""prior"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit` (like ""most_frequent""). ``predict_proba`` always returns the empirical class distribution of `y` also known as the empirical class prior distribution.* ""stratified"": the `predict_proba` method randomly samples one-hot vectors from a multinomial distribution parametrized by the empirical class prior probabilities. The `predict` method returns the class label which got probability one in the one-hot vector of `predict_proba`. Each sampled row of both methods is therefore independent and identically distributed.* ""uniform"": generates predictions uniformly at random from the list of unique classes observed in `y`, i.e. each class has equal probability.* ""constant"": always predicts a constant label that is provided by the user. This is useful for metrics that evaluate a non-majority class. .. versionchanged:: 0.24 The default value of `strategy` has changed to ""prior"" in version 0.24.",'most_frequent'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness to generate the predictions when``strategy='stratified'`` or ``strategy='uniform'``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"constant constant: int or str or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None


In [72]:
dummy_clsf.fit(train_dataset["text"], train_dataset["label"])

,"strategy strategy: {""most_frequent"", ""prior"", ""stratified"", ""uniform"", ""constant""}, default=""prior""Strategy to use to generate predictions.* ""most_frequent"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit`. The `predict_proba` method returns the matching one-hot encoded vector.* ""prior"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit` (like ""most_frequent""). ``predict_proba`` always returns the empirical class distribution of `y` also known as the empirical class prior distribution.* ""stratified"": the `predict_proba` method randomly samples one-hot vectors from a multinomial distribution parametrized by the empirical class prior probabilities. The `predict` method returns the class label which got probability one in the one-hot vector of `predict_proba`. Each sampled row of both methods is therefore independent and identically distributed.* ""uniform"": generates predictions uniformly at random from the list of unique classes observed in `y`, i.e. each class has equal probability.* ""constant"": always predicts a constant label that is provided by the user. This is useful for metrics that evaluate a non-majority class. .. versionchanged:: 0.24 The default value of `strategy` has changed to ""prior"" in version 0.24.",'most_frequent'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness to generate the predictions when``strategy='stratified'`` or ``strategy='uniform'``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"constant constant: int or str or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
Name,Type,Value
"class_prior_ class_prior_: ndarray of shape (n_classes,) or list of such arraysFrequency of each class observed in `y`. For multioutput classificationproblems, this is computed independently for each output.","ndarray[float64](6,)","[0.29,0.33,0.08,0.13,0.12,0.04]"
"classes_ classes_: ndarray of shape (n_classes,) or list of such arraysUnique class labels observed in `y`. For multi-output classificationproblems, this attribute is a list of arrays as each output has anindependent set of possible classes.","ndarray[int64](6,)","[0,1,2,3,4,5]"
n_classes_ n_classes_: int or list of intNumber of label for each output.,int,6
n_outputs_ n_outputs_: intNumber of outputs.,int,1
sparse_output_ sparse_output_: boolTrue if the array returned from predict is to be in sparse CSC format.Is automatically set to True if the input `y` is passed in sparseformat.,bool,False


In [78]:
f1_score(y_pred = dummy_clsf.predict(train_dataset["text"]).tolist(),
         y_true = train_dataset["label"].tolist(), average = "macro")

0.0836423955056883

### Tokenization and dataset preparation for RNN

In [ ]:
# Create tokeinzer object from HF tokenizers
tokenizer = Tokenizer(model = BPE(unk_token="[UNK]"))

# Trainer tokenizer
trainer = BpeTrainer(
    vocab_size =200,  # Desired vocabulary size
    min_frequency= 1,   # Minimum frequency for a token to be included
    special_tokens=["[PAD]", "[UNK]", "[CLS]"]
)
tokenizer.normalizer = Lowercase()
tokenizer.decoder = BPEDecoder(suffix = "#")

In [22]:
train_dataset["text"][:5]

0                              i didnt feel humiliated
1    i can go from feeling so hopeless to so damned...
2     im grabbing a minute to post i feel greedy wrong
3    i am ever feeling nostalgic about the fireplac...
4                                 i am feeling grouchy
Name: text, dtype: str

In [23]:
tokenizer.train_from_iterator(train_dataset["text"], trainer = trainer)

In [24]:
vocab = tokenizer.get_vocab()
print(f"Length of vocabulary: {len(vocab)}")
vocab

Length of vocabulary: 200


{'ke ': 84,
 'c': 6,
 'be': 82,
 'or': 50,
 'o': 18,
 'ev': 116,
 'ch': 86,
 'ent': 182,
 'i feel ': 61,
 'ar': 60,
 'ed ': 52,
 'f ': 59,
 'ow': 79,
 'ough': 185,
 'es ': 104,
 'bu': 103,
 'an ': 111,
 'ri': 101,
 'are ': 194,
 'just ': 165,
 'for ': 115,
 'di': 110,
 'wa': 83,
 '[CLS]': 2,
 'o ': 43,
 'y ': 38,
 'f': 9,
 'om': 75,
 'no': 112,
 'as': 113,
 'st ': 196,
 'a ': 62,
 'ic': 155,
 'll ': 140,
 'i c': 158,
 'e': 8,
 'i have ': 198,
 'at ': 57,
 'ing': 42,
 'ch ': 146,
 'k ': 96,
 'al ': 181,
 'en': 48,
 'ow ': 109,
 'feeling ': 69,
 'and ': 51,
 'ra': 147,
 'k': 14,
 'le': 88,
 'si': 135,
 'el': 37,
 'li': 64,
 'con': 180,
 'it ': 87,
 'at': 91,
 'm ': 58,
 'r': 21,
 'on': 47,
 'v': 25,
 'feel': 41,
 'is ': 74,
 'ant ': 157,
 'e ': 30,
 'w': 26,
 'out ': 102,
 'da': 148,
 'h': 11,
 'z': 29,
 'wor': 189,
 'ed': 143,
 'i feel like ': 177,
 'because ': 195,
 'im ': 97,
 'it': 90,
 'p ': 117,
 'my ': 80,
 'd': 7,
 'when ': 170,
 've ': 72,
 'ld ': 128,
 'fu': 142,
 'the': 131,
 

In [25]:
# CLS token
vocab["[CLS]"]

2

In [26]:
main_tokenizer = PreTrainedTokenizerFast(tokenizer_object = tokenizer, pad_token = "[PAD]")

In [27]:
# Encoding with cls token
input_ids = main_tokenizer("Hi there! Never you mind that", padding = "max_length", max_length = 50, trucation = True)['input_ids']
input_ids

[11,
 35,
 32,
 45,
 8,
 1,
 3,
 17,
 116,
 67,
 167,
 16,
 33,
 34,
 32,
 91,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

In [28]:
main_tokenizer.decode(input_ids, skip_special_tokens = True)

'hi there never you mind that'

In [29]:
max_number_of_tokens = 0
for text in train_dataset["text"]:
    tokenized_text = main_tokenizer(text)['input_ids']
    max_number_of_tokens = max(max_number_of_tokens, len(tokenized_text))
print(f"Maxium number of tokens in a single sentence: {max_number_of_tokens}")

Maxium number of tokens in a single sentence: 160


In [391]:
# Create torch datasets
class EmotionDataset(Dataset):
    def __init__(self, data: pd.DataFrame, 
                 tokenizer: PreTrainedTokenizerFast, max_length: int = 160):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __getitem__(self, idx) -> Tuple[torch.tensor, int]:
        single_row = self.data.iloc[idx]
        raw_text = single_row["text"]
        tokenized_text = self.tokenizer(raw_text, padding = "max_length",
                                        max_length = self.max_length, truncation = False,
                                        return_tensors = "pt")["input_ids"]
        return tokenized_text.squeeze(dim = 0), single_row["label"].item(), len(self.tokenizer(raw_text)["input_ids"])

    def __len__(self):
        return len(self.data)

In [392]:
main_tokenizer("hello, my name is luchian")["input_ids"]

[11, 37, 107, 1, 3, 80, 17, 4, 95, 74, 15, 24, 86, 12, 39]

In [405]:
# Random emotions dataset item
emotion_dataset = EmotionDataset(train_dataset, tokenizer = main_tokenizer)
random_item = emotion_dataset[rd.randint(0, len(emotion_dataset) - 1)]
print(random_item)
print(f"Random item tensor shape: {random_item[0].shape}")
random_item[0].shape

(tensor([179,  69,  26,  21,  47,  10,  52,  51,  12,  16,  19,  18,  23, 182,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0]), 3, 14)
Random item tensor shape: torch.Size([160])


torch.Size([160])

In [394]:
# Decoding random example
main_tokenizer.decode(random_item[0].tolist(), skip_special_tokens = False)

'i am feeling pressured and backed into a corner[PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD]'

In [267]:
emotions_dataset_demo = EmotionDataset(train_dataset.iloc[:5, :], tokenizer = main_tokenizer)
print(len(emotions_dataset_demo))

5


In [426]:
# Prepare all the other datasets
emotions_dataset_train = EmotionDataset(train_dataset, tokenizer = main_tokenizer)
emotions_dataset_val = EmotionDataset(val_dataset, tokenizer = main_tokenizer)
emotions_dataset_test = EmotionDataset(test_dataset, tokenizer = main_tokenizer)
print(f"Lengths of datasets: {len(emotions_dataset_train)}, {len(emotions_dataset_val)}, {len(emotions_dataset_test)}")

Lengths of datasets: 15956, 1988, 1986


In [427]:
emotions_dataset_train[9]

(tensor([ 61,  21,  75,  39,  63,   6,   3, 164,  18,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0]),
 2,
 9)

In [440]:
# Create emotion model
class EmotionModel(nn.Module):
    def __init__(self, input_size: int, hidden_size: int,
                 embedding_num: int, pad_id: int = 0, num_layers: int = 1, num_cls: int = 6):
        super().__init__()
        self.embed = nn.Embedding(embedding_num, input_size, padding_idx = pad_id)
        self.linear = nn.Linear(hidden_size, num_cls)
        self.rnn = nn.RNN(input_size, hidden_size, batch_first = False,
                          num_layers = num_layers, nonlinearity = "relu")
    
    def forward(self, x, lengths):
        y = self.embed(x)
        y = y.transpose(0, 1)
        y = pack_padded_sequence(y, lengths = lengths, batch_first = False, enforce_sorted=False)
        out, h = self.rnn(y)
        y = h.squeeze(dim = 0)
        y = self.linear(y)
        return y

In [416]:
def inference(text: str, model: nn.Module) -> Tuple[int, str]:
    pass

In [417]:
@dataclass
class TrainConfig:
    batch_size: int = 32
    learning_rate: float = 2e-3
    betas: Tuple[float, float] = (0.99, 0.98)
    epoch: int = 1000

In [418]:
config = TrainConfig()
config

TrainConfig(batch_size=32, learning_rate=0.002, betas=(0.99, 0.98), epoch=1000)

In [428]:
train_dataloader = DataLoader(dataset = emotions_dataset_train,
                              shuffle = True, batch_size = config.batch_size)

In [429]:
val_loader = DataLoader(dataset = emotions_dataset_val, shuffle = False, batch_size = config.batch_size)
val_loader

In [433]:
train_sample = next(iter(train_dataloader))
print(train_sample)
print(f"Shape tensor of train sample: {train_sample[0].shape}")

[tensor([[ 35,  32,  33,  ...,   0,   0,   0],
        [ 61, 129, 141,  ...,   0,   0,   0],
        [ 61, 100,  11,  ...,   0,   0,   0],
        ...,
        [ 35,  65,  23,  ...,   0,   0,   0],
        [ 35, 110,  22,  ...,   0,   0,   0],
        [ 61,  23,  48,  ...,   0,   0,   0]]), tensor([5, 0, 1, 2, 1, 0, 1, 3, 1, 0, 0, 2, 0, 0, 1, 5, 1, 0, 1, 3, 2, 0, 3, 1,
        0, 0, 3, 0, 0, 4, 1, 2]), tensor([101,  37,  18,  44,  22,  17,  44,  10,  27,  19,  28,  26,  27,  30,
         61,  86,  37,  35,  58,  61,  48,  40,  35,  16,  39,  21,  60,  51,
         15,  16,  55,  15])]
Shape tensor of train sample: torch.Size([32, 160])


In [434]:
len(main_tokenizer.get_vocab())

200

In [441]:
rnn_model = EmotionModel(input_size = 300,
                         hidden_size = 300,
                         embedding_num = len(main_tokenizer.get_vocab())).to(device = 'cuda')
rnn_model

EmotionModel(
  (embed): Embedding(200, 300, padding_idx=0)
  (linear): Linear(in_features=300, out_features=6, bias=True)
  (rnn): RNN(300, 300)
)

In [442]:
rnn_model(train_sample[0].to(device = 'cuda'), lengths = train_sample[2]).shape

torch.Size([32, 6])

In [466]:
# Training loop for model_training
def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
    save: bool = False,
    save_epoch: int = 0,
    base_path: str = '',
    save_model_name: str = '',
    save_optimizer_name : str = ''
) -> float:
    """Run one training epoch."""
    if save:
        torch.save(model.state_dict(), base_path + f"Epoch #{save_epoch}" + "-" + save_model_name)
        torch.save(optimizer.state_dict(), base_path + f"Epoch #{save_epoch}" + "-" + save_optimizer_name)
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    for inputs, targets, lengths in tqdm(dataloader, desc="Epoch training"):
        inputs, targets, lengths = inputs.to(device), targets.to(device), lengths.to(device = 'cpu')
        optimizer.zero_grad()

        outputs = model(inputs, lengths = lengths)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [445]:
example_input = emotions_dataset_train.tokenizer("The waiters at this restaurant are awful", return_tensors = "pt")["input_ids"]
example_input

tensor([[ 54,  83,  90,  45,  36,  57, 129,  56,  68,   4, 171, 157, 194,   4,
          26, 142,  15]])

In [467]:
# Given the model calculate f1-score on validation dataset
def calculate_val_score(dataloader: DataLoader,
                        model: nn.Module, device: str = "cuda") -> float:
    model.eval()
    y_preds = []
    y_trues = []
    for input_tensors, labels, lengths in dataloader:
        input_tensors, labels, lengths = input_tensors.to(device = device), labels.to(device = device), lengths.to(device = 'cpu')
        predictions = model(input_tensors, lengths = lengths)
        y_pred = predictions.softmax(dim = 1).argmax(dim = 1).detach().tolist()
        y_true = labels.detach().tolist()
        y_preds.extend(y_pred)
        y_trues.extend(y_true)
    return f1_score(y_true = y_trues, y_pred = y_preds, average = "macro")

### TF-IDF + Linear

In [82]:
tf_idf_vectorizer = TfidfVectorizer()
tf_idf_vectorizer 

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'
,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1

In [84]:
sparse_array = tf_idf_vectorizer.fit_transform(train_dataset["text"])
sparse_array

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 249139 stored elements and shape (15956, 15184)>

In [87]:
vectorized_dataset = sparse_array.toarray()
vectorized_dataset

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(15956, 15184))

In [92]:
type(train_dataset["label"])
train_dataset["label"]

0        0
1        0
2        3
3        2
4        3
        ..
15951    0
15952    0
15953    1
15954    3
15955    0
Name: label, Length: 15956, dtype: int64

In [95]:
train_dataset["label"].iloc[1].item()

0

In [96]:
vectorized_dataset[0]

array([0., 0., 0., ..., 0., 0., 0.], shape=(15184,))

In [145]:
class SimpleLinearDataset(Dataset):
    def __init__(self, vectorized_dataset: np.array, labels: pd.Series):
        self.dataset = vectorized_dataset
        self.labels = labels

    def __getitem__(self, idx):
        item, label = torch.from_numpy(self.dataset[idx]), self.labels.iloc[idx].item()
        return item.to(dtype=torch.float32), label

    def __len__(self):
        return len(self.dataset)

In [183]:
class SimpleLinearDatasetVal(Dataset):
    def __init__(self, text_corpus: pd.Series, labels: pd.Series, vectorizer: callable):
        self.dataset = text_corpus
        self.labels = labels
        self.vectorizer = vectorizer

    def __getitem__(self, idx):
        item, label = torch.from_numpy(self.vectorizer.transform([self.dataset.iloc[idx]]).toarray()).to(dtype=torch.float32).squeeze(dim = 0), self.labels.iloc[idx]
        return item, label

    def __len__(self):
        return len(self.dataset)

In [168]:
simple_dataset = SimpleLinearDataset(vectorized_dataset, train_dataset["label"])
simple_dataset

In [163]:
# simple sample from the dataset
simple_dataset[0]

(tensor([0., 0., 0.,  ..., 0., 0., 0.]), 0)

In [184]:
simple_val_dataset = SimpleLinearDatasetVal(train_dataset["text"], train_dataset["label"], tf_idf_vectorizer)

In [185]:
# simple sample from the validation dataset
simple_val_dataset[0]

(tensor([0., 0., 0.,  ..., 0., 0., 0.]), np.int64(0))

In [186]:
train_dataloader = DataLoader(dataset = simple_dataset,
                              shuffle = True, batch_size = config.batch_size)

In [187]:
# Train loader sample
next(iter(train_dataloader))

[tensor([[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]),
 tensor([4, 3, 0, 3, 1, 1, 4, 0, 1, 2, 0, 0, 3, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1,
         0, 3, 3, 0, 1, 3, 3, 0])]

In [188]:
val_dataloader = DataLoader(dataset = simple_val_dataset,
                            shuffle = True, batch_size = config.batch_size)

In [190]:
# val loader sample
next(iter(val_dataloader))

[tensor([[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]),
 tensor([1, 0, 4, 2, 3, 1, 3, 1, 0, 0, 1, 0, 0, 4, 4, 4, 0, 1, 1, 1, 0, 1, 0, 0,
         4, 1, 0, 0, 0, 0, 1, 2])]

In [191]:
tfidf_model = nn.Linear(15184, 6).to(device = 'cuda', dtype=torch.float32)

In [192]:
optimizer = optim.AdamW(params = tfidf_model.parameters(), lr = config.learning_rate)
criterion = nn.CrossEntropyLoss(ignore_index = 0)

In [194]:
# Training looop 
for ep in tqdm(range(2_000), desc = "Current epoch"):
    if (ep+1) % 10 == 0:
        current_loss = train_one_epoch(model = tfidf_model,
                        dataloader = train_dataloader,
                        optimizer = optimizer,
                        criterion = criterion,
                        device = 'cuda',
                        save = True,
                        save_epoch = ep + 1,
                        base_path= "artifacts\\",
                        save_model_name="SimpleLinear.pth",
                        save_optimizer_name="SimpleLinearOptimizer.pth")
    else:
        current_loss = train_one_epoch(model = tfidf_model,
                        dataloader = train_dataloader,
                        optimizer = optimizer,
                        criterion = criterion,
                        device = 'cuda',
                        save = False) 
    current_val = calculate_val_score(val_dataloader, tfidf_model, device = "cuda")
    print(f"Epoch # {ep + 1} | Train loss: {current_loss} | Val f1_score: {current_val}")
    

Current epoch:   0%|          | 1/2000 [00:09<5:20:57,  9.63s/it]

Epoch # 1 | Train loss: 1.1652699351788522 | Val f1_score: 0.20050224383971504


Current epoch:   0%|          | 2/2000 [00:19<5:17:48,  9.54s/it]

Epoch # 2 | Train loss: 1.0085509477970833 | Val f1_score: 0.3225424746607725


Current epoch:   0%|          | 3/2000 [00:28<5:20:07,  9.62s/it]

...

Epoch # 44 | Train loss: 0.058848757561826275 | Val f1_score: 0.715867127086697


Current epoch:   2%|▏         | 45/2000 [07:49<5:22:10,  9.89s/it]

Epoch # 45 | Train loss: 0.05711707018711464 | Val f1_score: 0.7141001219674981


Current epoch:   2%|▏         | 45/2000 [07:49<5:40:09, 10.44s/it]


KeyboardInterrupt: 

In [ ]:
# Saving artifacts
# with open("artifacts\\SimpleLinearTFIDFVectorizer.pkl", 'wb') as file:
#     pickle.dump(tf_idf_vectorizer, file)

In [208]:
# Saving artifacts
# with open("artifacts\\SimpleLinearTFIDFVectorizer.pkl", 'rb') as file:
#     obj  = pickle.load(file)

### RRN training

In [459]:
@dataclass
class TrainConfig:
    batch_size: int = 32
    learning_rate: float = 2e-3
    betas: Tuple[float, float] = (0.99, 0.98)
    epoch: int = 1000

In [460]:
config = TrainConfig()

In [461]:
train_dataloader = DataLoader(dataset = emotions_dataset_train,
                              shuffle = True, batch_size = config.batch_size)

In [462]:
val_loader = DataLoader(dataset = emotions_dataset_val, shuffle = False, batch_size = config.batch_size)
val_loader

In [463]:
rnn_model = EmotionModel(input_size = 300,
                         hidden_size = 150,
                         embedding_num = len(main_tokenizer.get_vocab())).to(device = 'cuda')

In [453]:
# rnn_model.load_state_dict(torch.load("C:\\main\\GitHub\\nlpTasks\\artifacts\Epoch #2000-SimpleRNN.pth"))

In [464]:
optimizer = optim.AdamW(params = rnn_model.parameters(), lr = config.learning_rate, betas = config.betas)
criterion = nn.CrossEntropyLoss(ignore_index = 0)

In [455]:
# optimizer.load_state_dict(torch.load("C:\\main\\GitHub\\nlpTasks\\artifacts\Epoch #2000-SimpleRNNOptimizer.pth"))

In [468]:
# Training looop 
for ep in tqdm(range(2_000), desc = "Current epoch"):
    if (ep+1) % 10 == 0:
        current_loss = train_one_epoch(model = rnn_model,
                        dataloader = train_dataloader,
                        optimizer = optimizer,
                        criterion = criterion,
                        device = 'cuda',
                        save = True,
                        save_epoch = ep + 1,
                        base_path= "artifacts\\",
                        save_model_name="SimpleRNN.pth",
                        save_optimizer_name="SimpleRNNOptimizer.pth")
    else:
        current_loss = train_one_epoch(model = rnn_model,
                        dataloader = train_dataloader,
                        optimizer = optimizer,
                        criterion = criterion,
                        device = 'cuda',
                        save = False) 
    current_val = calculate_val_score(val_loader, rnn_model, device = "cuda")
    print(f"Epoch # {ep + 1} | Train loss: {current_loss} | Val f1_score: {current_val}")
    

Current epoch:   0%|          | 1/2000 [00:14<8:04:06, 14.53s/it]

Epoch # 1 | Train loss: 1.4040698513000427 | Val f1_score: 0.09935834234683223


Current epoch:   0%|          | 2/2000 [00:27<7:31:18, 13.55s/it]

Epoch # 2 | Train loss: 1.3516172015356396 | Val f1_score: 0.1354272252357986


Current epoch:   0%|          | 3/2000 [00:40<7:19:29, 13.20s/it]

...

Epoch # 207 | Train loss: nan | Val f1_score: 0.07223535592329919


Current epoch:  10%|█         | 208/2000 [43:48<6:21:27, 12.77s/it]

Epoch # 208 | Train loss: nan | Val f1_score: 0.07223535592329919


Current epoch:  10%|█         | 208/2000 [43:50<6:17:42, 12.65s/it]


KeyboardInterrupt: 

### LSTM training

In [59]:
some_string = 'Here there is going to be LSTM training'

### GRU training

In [58]:
some_string = "Here is going to be GRU training"

### BERT training

In [60]:
some_string = "Here is going to be BERT training"